# <center>**Chapter 21 - Managing Training Process with Checkpoints and Early Stopping**<center>

## **Mục tiêu**
1. Lưu trạng thái mạng, optimizer và epoch để có thể tiếp tục một quá trình huấn luyện bị gián đoạn.
2. Theo dõi validation loss, lưu trạng thái tốt nhất và dừng khi không còn cải thiện.
3. Chỉ đánh giá tập test ở bước cuối, không dùng test để điều khiển huấn luyện.

Dùng Pima có sẵn trong `code/data`, không cần tải dữ liệu bên ngoài. Tệp checkpoint tạm sẽ được tạo trong thư mục dữ liệu của buổi thực hành và xóa ở cuối notebook.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch import nn, optim
from torch.utils.data import DataLoader, TensorDataset

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(1)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [2]:
def find_pima_file() -> Path:
    # Tìm dữ liệu khi kernel chạy từ gốc dự án hoặc từ thư mục notebook.
    lab_dir_name = "Buổi thực hành số 7 (Optimizing DL Networks)"
    current = Path.cwd().resolve()
    for root in (current, *current.parents):
        candidates = (
            root / "data" / "pima-indians-diabetes.csv",
            root / "code" / "data" / "pima-indians-diabetes.csv",
            root / lab_dir_name / "code" / "data" / "pima-indians-diabetes.csv",
        )
        for candidate in candidates:
            if candidate.is_file():
                return candidate
    raise FileNotFoundError("Không tìm thấy pima-indians-diabetes.csv trong code/data.")

pima_path = find_pima_file()
X_frame = pd.read_csv(pima_path).iloc[:, :-1].copy()
y = pd.read_csv(pima_path).iloc[:, -1].to_numpy(dtype=np.int64)
# Một số số 0 ở chỉ số y tế là giá trị thiếu; không thay số lần mang thai.
columns_with_missing_zero = X_frame.columns[[1, 2, 3, 4, 5]]
X_frame[columns_with_missing_zero] = X_frame[columns_with_missing_zero].astype(float).replace(0, np.nan)

In [3]:
X_train_valid, X_test, y_train_valid, y_test = train_test_split(
    X_frame.to_numpy(dtype=np.float32),
    y,
    test_size=0.2,
    stratify=y,
    random_state=SEED,
)
X_train, X_valid, y_train, y_valid = train_test_split(
    X_train_valid,
    y_train_valid,
    test_size=0.25,
    stratify=y_train_valid,
    random_state=SEED,
)

imputer = SimpleImputer(strategy="median")
scaler = StandardScaler()
X_train = scaler.fit_transform(imputer.fit_transform(X_train)).astype(np.float32)
X_valid = scaler.transform(imputer.transform(X_valid)).astype(np.float32)
X_test = scaler.transform(imputer.transform(X_test)).astype(np.float32)

In [4]:
train_dataset = TensorDataset(
    torch.from_numpy(X_train),
    torch.from_numpy(y_train.astype(np.float32).reshape(-1, 1)),
)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
X_valid_tensor = torch.from_numpy(X_valid).to(DEVICE)
y_valid_tensor = torch.from_numpy(y_valid.astype(np.float32).reshape(-1, 1)).to(DEVICE)
X_test_tensor = torch.from_numpy(X_test).to(DEVICE)
y_test_tensor = torch.from_numpy(y_test.astype(np.float32).reshape(-1, 1)).to(DEVICE)
data_dir = pima_path.parent
checkpoint_path = data_dir / ".chapter21_checkpoint.pth"

print(f"Mẫu train/validation/test: {len(X_train)}/{len(X_valid)}/{len(X_test)}")
print("Thiết bị:", DEVICE)

Mẫu train/validation/test: 460/154/154
Thiết bị: cpu


## **Checkpoint và tiếp tục huấn luyện**

Ngoài trọng số mô hình, checkpoint lưu trạng thái optimizer (ví dụ các moment của Adam) và epoch tiếp theo. Ví dụ mô phỏng dừng sau 5 epoch, khởi tạo lại mô hình/optimizer, nạp checkpoint rồi tiếp tục.

In [5]:
class DiabetesClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(8, 16),
            nn.ReLU(),
            nn.Dropout(p=0.2),
            nn.Linear(16, 1),
        )

    def forward(self, features: torch.Tensor) -> torch.Tensor:
        return self.network(features)

loss_fn = nn.BCEWithLogitsLoss()

def save_checkpoint(
    model: nn.Module,
    optimizer: optim.Optimizer,
    next_epoch: int,
    validation_loss: float,
) -> None:
    # Lưu cả trạng thái optimizer để tiếp tục cập nhật từ đúng trạng thái trước đó.
    torch.save(
        {
            "next_epoch": next_epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "validation_loss": validation_loss,
        },
        checkpoint_path,
    )

def load_checkpoint(model: nn.Module, optimizer: optim.Optimizer) -> int:
    checkpoint = torch.load(
        checkpoint_path,
        map_location=DEVICE,
        weights_only=True,
    )
    model.load_state_dict(checkpoint["model_state_dict"])
    optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
    return int(checkpoint["next_epoch"])

def train_epochs(
    model: nn.Module,
    optimizer: optim.Optimizer,
    start_epoch: int,
    end_epoch: int,
) -> None:
    for epoch in range(start_epoch, end_epoch):
        model.train()
        for features, labels in train_loader:
            features, labels = features.to(DEVICE), labels.to(DEVICE)
            loss = loss_fn(model(features), labels)
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            optimizer.step()

        model.eval()
        with torch.inference_mode():
            valid_loss = loss_fn(model(X_valid_tensor), y_valid_tensor).item()
        save_checkpoint(model, optimizer, epoch + 1, valid_loss)
        print(f"Epoch {epoch + 1:02d}: validation loss = {valid_loss:.4f}")

In [6]:
model = DiabetesClassifier().to(DEVICE)
optimizer = optim.Adam(model.parameters(), lr=1e-3)
train_epochs(model, optimizer, start_epoch=0, end_epoch=5)

resumed_model = DiabetesClassifier().to(DEVICE)
resumed_optimizer = optim.Adam(resumed_model.parameters(), lr=1e-3)
next_epoch = load_checkpoint(resumed_model, resumed_optimizer)
print(f"Khôi phục thành công; epoch tiếp theo: {next_epoch + 1}.")
train_epochs(resumed_model, resumed_optimizer, start_epoch=next_epoch, end_epoch=10)

Epoch 01: validation loss = 0.7891
Epoch 02: validation loss = 0.7516
Epoch 03: validation loss = 0.7191
Epoch 04: validation loss = 0.6889
Epoch 05: validation loss = 0.6620
Khôi phục thành công; epoch tiếp theo: 6.
Epoch 06: validation loss = 0.6364
Epoch 07: validation loss = 0.6129
Epoch 08: validation loss = 0.5908
Epoch 09: validation loss = 0.5713
Epoch 10: validation loss = 0.5526


## **Early Stopping với validation loss**

Sau mỗi epoch, nếu validation loss thấp hơn tốt nhất trước đó thì lưu checkpoint và đặt lại bộ đếm. Nếu không cải thiện trong `patience` epoch liên tiếp, vòng lặp dừng và nạp lại mô hình tốt nhất. Tập test chỉ được dùng sau khi đã chọn mô hình.

In [7]:
best_model = DiabetesClassifier().to(DEVICE)
best_optimizer = optim.Adam(best_model.parameters(), lr=1e-3)
best_valid_loss = float("inf")
best_epoch = 0
epochs_without_improvement = 0
patience = 8
max_epochs = 100

for epoch in range(max_epochs):
    best_model.train()
    for features, labels in train_loader:
        features, labels = features.to(DEVICE), labels.to(DEVICE)
        loss = loss_fn(best_model(features), labels)
        best_optimizer.zero_grad(set_to_none=True)
        loss.backward()
        best_optimizer.step()

    best_model.eval()
    with torch.inference_mode():
        valid_loss = loss_fn(best_model(X_valid_tensor), y_valid_tensor).item()

    if valid_loss < best_valid_loss:
        best_valid_loss = valid_loss
        best_epoch = epoch + 1
        epochs_without_improvement = 0
        save_checkpoint(best_model, best_optimizer, epoch + 1, valid_loss)
    else:
        epochs_without_improvement += 1

    print(
        f"Epoch {epoch + 1:02d}: validation loss = {valid_loss:.4f}; "
        f"tốt nhất = {best_valid_loss:.4f}"
    )
    if epochs_without_improvement >= patience:
        print(f"Dừng sớm sau {epoch + 1} epoch (patience={patience}).")
        break

Epoch 01: validation loss = 0.6936; tốt nhất = 0.6936
Epoch 02: validation loss = 0.6699; tốt nhất = 0.6699
Epoch 03: validation loss = 0.6494; tốt nhất = 0.6494
Epoch 04: validation loss = 0.6321; tốt nhất = 0.6321
Epoch 05: validation loss = 0.6162; tốt nhất = 0.6162
Epoch 06: validation loss = 0.6007; tốt nhất = 0.6007
Epoch 07: validation loss = 0.5859; tốt nhất = 0.5859
Epoch 08: validation loss = 0.5728; tốt nhất = 0.5728
Epoch 09: validation loss = 0.5594; tốt nhất = 0.5594
Epoch 10: validation loss = 0.5466; tốt nhất = 0.5466
Epoch 11: validation loss = 0.5351; tốt nhất = 0.5351
Epoch 12: validation loss = 0.5232; tốt nhất = 0.5232
Epoch 13: validation loss = 0.5130; tốt nhất = 0.5130
Epoch 14: validation loss = 0.5029; tốt nhất = 0.5029
Epoch 15: validation loss = 0.4937; tốt nhất = 0.4937
Epoch 16: validation loss = 0.4848; tốt nhất = 0.4848
Epoch 17: validation loss = 0.4763; tốt nhất = 0.4763
Epoch 18: validation loss = 0.4683; tốt nhất = 0.4683
Epoch 19: validation loss = 

In [8]:
load_checkpoint(best_model, best_optimizer)
best_model.eval()
with torch.inference_mode():
    test_probabilities = torch.sigmoid(best_model(X_test_tensor))
    test_predictions = (test_probabilities >= 0.5).cpu().numpy().ravel()

print(f"Epoch tốt nhất: {best_epoch}; validation loss: {best_valid_loss:.4f}")
print(f"Accuracy test (đánh giá cuối): {accuracy_score(y_test, test_predictions):.3f}")

Epoch tốt nhất: 55; validation loss: 0.4155
Accuracy test (đánh giá cuối): 0.714


In [9]:
# Dọn checkpoint tạm để không để lại tệp phát sinh sau khi chạy notebook.
if checkpoint_path.exists():
    checkpoint_path.unlink()

## **Kết luận**

Checkpoint giúp phục hồi cả trọng số và trạng thái optimizer; early stopping tránh tiếp tục huấn luyện khi validation không còn cải thiện. Trong bài toán thực tế, giữ riêng tập test và chỉ dùng kết quả của tập này để đánh giá cuối cùng.

# <center>**HẾT**<center>